# Lezione 4 · Ottimizzazione I: come impara una rete

**Corso: Advanced Deep Learning**

Questo quaderno è quello che eseguirai dal vivo durante la lezione. Combina spiegazioni brevi, in stile appunti, con celle di codice che esegui e modifichi al momento. L'obiettivo non è memorizzare formule, ma costruire intuizione sull'algoritmo alla base di ogni addestramento di una rete: la discesa del gradiente (*gradient descent*, GD).

## Obiettivi di apprendimento

Al termine di questa lezione saprai:

- Spiegare che cos'è una funzione obiettivo (o funzione di perdita) e perché addestrare una rete equivale a minimizzarla
- Implementare la discesa del gradiente a mano, senza alcun framework, usando solo NumPy
- Spiegare come il tasso di apprendimento determina se l'addestramento converge, diverge o oscilla
- Riconoscere il numero di condizionamento di una superficie di perdita e spiegare perché un valore alto complica l'addestramento
- Collegare la teoria (costante di Lipschitz, passo massimo stabile $1/L$) a ciò che osservi nei tuoi grafici

## 1. La funzione obiettivo come un paesaggio

Immagina di trovarti in cima a una montagna, avvolto da una nebbia fitta. Vedi solo il terreno sotto i tuoi piedi. Il tuo obiettivo è raggiungere il punto più basso della valle, e non hai una mappa: in ogni punto percepisci verso dove il terreno scende di più e fai un passo in quella direzione.

Questa è l'idea della discesa del gradiente. La "montagna" è la funzione obiettivo $h(\theta)$: misura quanto si sbaglia il tuo modello per ogni combinazione di parametri $\theta$. La tua "posizione" sono i valori correnti di quei parametri. Il "senso della pendenza" è il gradiente $\nabla h(\theta)$, un vettore che punta verso dove la funzione cresce più rapidamente. Poiché vuoi scendere, ti muovi nella direzione opposta: il gradiente negativo.

Disegna per prima cosa questo paesaggio con una semplice funzione di perdita quadratica a due parametri, $\theta_1$ e $\theta_2$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# h(theta) = 0.5 * (a * theta_1^2 + b * theta_2^2)
# 'a' y 'b' son las curvaturas en cada dirección del paisaje
a, b = 1.0, 5.0

def loss(theta):
    return 0.5 * (a * theta[0] ** 2 + b * theta[1] ** 2)

theta1 = np.linspace(-6, 6, 200)
theta2 = np.linspace(-6, 6, 200)
T1, T2 = np.meshgrid(theta1, theta2)
Z = 0.5 * (a * T1 ** 2 + b * T2 ** 2)

fig = plt.figure(figsize=(13, 5))

# Panel izquierdo: curvas de nivel, la vista "desde arriba" de la montaña
ax1 = fig.add_subplot(1, 2, 1)
ax1.contour(T1, T2, Z, levels=20, cmap="viridis")
ax1.plot(0, 0, "r*", markersize=15, label="Mínimo")
ax1.set_xlabel("theta_1")
ax1.set_ylabel("theta_2")
ax1.set_title("Curvas de nivel de h(theta)")
ax1.legend()

# Panel derecho: la superficie 3D, la "montaña" real de la analogía
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
ax2.plot_surface(T1, T2, Z, cmap="viridis", alpha=0.85)
ax2.set_xlabel("theta_1")
ax2.set_ylabel("theta_2")
ax2.set_zlabel("h(theta)")
ax2.set_title("La función objetivo como paisaje")

plt.tight_layout()
plt.show()

## 2. Discesa del gradiente passo dopo passo

La regola di aggiornamento della discesa del gradiente è:

$$\theta_{k+1} = \theta_k - \alpha \, \nabla h(\theta_k)$$

A ogni iterazione $k$ calcoli il gradiente nella tua posizione attuale e fai un passo di ampiezza $\alpha$ (il tasso di apprendimento) nella direzione opposta. Ripeti il processo finché ti avvicini al minimo.

Implementa questa regola a mano, senza alcun framework, e salva l'intera traiettoria di $\theta$ per poterla disegnare sopra il paesaggio appena visto.

In [ ]:
def gradient(theta):
    # Derivadas parciales de la cuadrática respecto a cada parámetro
    return np.array([a * theta[0], b * theta[1]])

def gradient_descent(theta_init, alpha, num_steps):
    theta = np.array(theta_init, dtype=float)
    trayectoria = [theta.copy()]
    for _ in range(num_steps):
        theta = theta - alpha * gradient(theta)  # regla de actualización de GD
        trayectoria.append(theta.copy())
        if np.any(np.abs(theta) > 1e6):  # evita desbordar el tipo float si diverge
            break
    return np.array(trayectoria)

theta_inicial = [4.0, 4.0]
trayectoria = gradient_descent(theta_inicial, alpha=0.15, num_steps=25)

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.contour(T1, T2, Z, levels=20, cmap="viridis", alpha=0.6)
ax.plot(trayectoria[:, 0], trayectoria[:, 1], "o-", color="crimson", markersize=4, label="Trayectoria de GD")
ax.plot(0, 0, "r*", markersize=15, label="Mínimo")
ax.set_xlabel("theta_1")
ax.set_ylabel("theta_2")
ax.set_title("Descenso de gradiente sobre el paisaje")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Punto de partida: {theta_inicial}")
print(f"Punto final tras {len(trayectoria) - 1} pasos: {trayectoria[-1]}")
print(f"Pérdida inicial: {loss(trayectoria[0]):.4f} -> Pérdida final: {loss(trayectoria[-1]):.6f}")

## 3. Il ruolo cruciale del tasso di apprendimento

Il tasso di apprendimento decide l'ampiezza del passo, e da esso dipende tutto:

- **Passo adeguato**: avanzi in modo costante e ragionevolmente rapido verso il minimo
- **Passo troppo grande**: salti oltre la valle e atterri più in alto di dove eri partito; se ripeti questo più volte, il processo diverge
- **Passo troppo piccolo**: avanzi in piena sicurezza, ma così lentamente che l'addestramento diventa impraticabile

La teoria ti offre un limite: con $L$ come curvatura massima della funzione (la costante di Lipschitz), un tasso $\alpha \le 1/L$ garantisce che ogni passo non aumenti la perdita. Confronta ora tre tassi diversi sullo stesso paesaggio.

In [ ]:
num_pasos = 40
L = max(a, b)
alpha_max = 1.0 / L  # tasa máxima teórica de estabilidad conservadora

tasas = {
    "Buena (alpha = 0.15)": 0.15,
    "Demasiado alta (alpha = 0.45)": 0.45,   # supera 2/L y el proceso diverge
    "Demasiado baja (alpha = 0.02)": 0.02,   # converge, pero muy despacio
}

resultados = {
    nombre: gradient_descent(theta_inicial, alpha, num_pasos)
    for nombre, alpha in tasas.items()
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel izquierdo: las tres trayectorias sobre el contour plot
ax = axes[0]
ax.contour(T1, T2, Z, levels=25, cmap="viridis", alpha=0.6)
for nombre, trayectoria in resultados.items():
    ax.plot(trayectoria[:, 0], trayectoria[:, 1], marker="o", markersize=3, label=nombre)
ax.plot(0, 0, "r*", markersize=15, label="Mínimo")
ax.set_xlim(-6, 6)
ax.set_ylim(-6, 6)
ax.set_xlabel("theta_1")
ax.set_ylabel("theta_2")
ax.set_title("Trayectorias según la tasa de aprendizaje")
ax.legend(fontsize=8)

# Panel derecho: evolución de la pérdida en escala logarítmica
ax = axes[1]
for nombre, trayectoria in resultados.items():
    valores_perdida = [loss(theta) for theta in trayectoria]
    ax.plot(valores_perdida, marker="o", markersize=3, label=nombre)
ax.set_yscale("log")
ax.set_xlabel("Iteración")
ax.set_ylabel("Pérdida h(theta) [escala log]")
ax.set_title("Convergencia según la tasa de aprendizaje")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"Tasa de aprendizaje máxima estable (1/L) = {alpha_max:.4f}")

## 4. Curvatura e numero di condizionamento

Ora complica il paesaggio: invece di una valle a forma di conca, immagina un canyon stretto e allungato, con pareti ripide ai lati e un fondo quasi piatto. Se fai passi grandi, rimbalzi da una parete all'altra senza quasi avanzare sul fondo: oscilli invece di progredire. Se riduci il passo per evitare il rimbalzo, avanzi in sicurezza lungo le pareti, ma ti trascini lungo il fondo piatto. Non esiste un unico tasso che risolva bene entrambe le direzioni insieme.

Il rapporto tra la curvatura massima e quella minima si chiama numero di condizionamento, $\kappa = L/\mu$. Più $\kappa$ è grande, più il canyon è stretto e più iterazioni servono: il numero di passi cresce in proporzione a $\kappa$. Aumenta ora il numero di condizionamento e osserva lo zigzag.

In [ ]:
a2, b2 = 1.0, 100.0  # numero de condicion kappa = b2 / a2 = 100: canon mucho mas estrecho
L2 = max(a2, b2)

def gradient2(theta):
    return np.array([a2 * theta[0], b2 * theta[1]])

def loss2(theta):
    return 0.5 * (a2 * theta[0] ** 2 + b2 * theta[1] ** 2)

def gradient_descent2(theta_init, alpha, num_steps):
    theta = np.array(theta_init, dtype=float)
    trayectoria = [theta.copy()]
    for _ in range(num_steps):
        theta = theta - alpha * gradient2(theta)
        trayectoria.append(theta.copy())
    return np.array(trayectoria)

theta_inicial2 = [4.0, 4.0]
alpha2 = 0.018  # una tasa "razonable" para la direccion plana, pero cercana al limite en la empinada
trayectoria2 = gradient_descent2(theta_inicial2, alpha2, num_steps=60)

theta1_r = np.linspace(-6, 6, 200)
theta2_r = np.linspace(-6, 6, 200)
T1b, T2b = np.meshgrid(theta1_r, theta2_r)
Zb = 0.5 * (a2 * T1b ** 2 + b2 * T2b ** 2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.contour(T1b, T2b, Zb, levels=30, cmap="viridis", alpha=0.6)
ax.plot(trayectoria2[:, 0], trayectoria2[:, 1], "o-", markersize=3, color="crimson")
ax.plot(0, 0, "r*", markersize=15, label="Mínimo")
ax.set_xlabel("theta_1 (curvatura baja)")
ax.set_ylabel("theta_2 (curvatura alta)")
ax.set_title(f"Zigzag con numero de condicion kappa = {b2 / a2:.0f}")
ax.legend()

ax = axes[1]
valores2 = [loss2(theta) for theta in trayectoria2]
ax.plot(valores2, "o-", markersize=3, color="crimson")
ax.set_yscale("log")
ax.set_xlabel("Iteración")
ax.set_ylabel("Pérdida h(theta) [escala log]")
ax.set_title("La pérdida baja, pero con muchas oscilaciones")

plt.tight_layout()
plt.show()

print(f"Numero de condicion (kappa) = {b2 / a2:.0f}")
print(f"Tasa de aprendizaje maxima estable (1/L) = {1 / L2:.4f}")
print(f"Perdida tras {len(trayectoria2) - 1} pasos: {loss2(trayectoria2[-1]):.4f} (compara con la seccion anterior)"
)

## Riepilogo dei punti chiave

- Addestrare una rete equivale a minimizzare una funzione obiettivo $h(\theta)$ muovendo i suoi parametri passo dopo passo
- La discesa del gradiente segue la regola $\theta_{k+1} = \theta_k - \alpha \nabla h(\theta_k)$: ti muovi in direzione opposta al gradiente, cioè la direzione di maggior crescita della perdita
- Il tasso di apprendimento $\alpha$ controlla l'ampiezza del passo: troppo alto e il processo oscilla o diverge; troppo basso e converge, ma molto lentamente
- La costante di Lipschitz $L$ limita la curvatura massima; un tasso $\alpha \le 1/L$ garantisce una stabilità prudente
- Il numero di condizionamento $\kappa = L/\mu$ misura quanto è più ripida la direzione più curvata rispetto a quella più piatta; più $\kappa$ è grande, più stretto è il "canyon" e più passi servono per convergere
- Un unico valore di $\alpha$ non risolve mai bene un canyon molto stretto: ciò che stabilizza la direzione ripida risulta lentissimo nella direzione piatta

**Apri ora il quaderno di pratica di accompagnamento, `../notebooks_practica/clase04_practica_optimizacion_fundamentos.ipynb`, e metti alla prova tu stesso queste idee.**